In [ ]:
# Leakage-prevention procedure for unstructured clinical notes
# Goal: remove note content documented after the prediction-time cutoff
# before LLM-based extraction of text-derived covariates.

import re
import pandas as pd
from datetime import datetime
from pathlib import Path

# 1. Define the latest allowable note timestamp per encounter.
# In the notebook, "time_lag_1 == True" marks the pre-target / prediction-eligible rows.
df["recorded_time"] = pd.to_datetime(df["recorded_time"])

cutoff_map = (
    df.loc[df["time_lag_1"] == True]
      .groupby("encounter_id")["recorded_time"]
      .max()
      .rename("recorded_time_max_no_target")
)

# Normalize timestamps for comparison with note timestamps, which are timezone-naive.
cutoff_map = cutoff_map.dt.tz_convert("UTC").dt.tz_localize(None)


# 2. Normalize encounter identifiers so dataframe rows and note filenames match.
def encounter_key(x):
    x = str(x).strip()
    if "/" in x:
        x = x.rsplit("/", 1)[-1]          # e.g., Encounter/<id> -> <id>
    x = re.sub(r"^(?i:Encounter[_:])", "", x)
    return x.lower()


cutoff_map.index = cutoff_map.index.map(encounter_key)
cutoff_map = cutoff_map.to_dict()


# 3. Detect timestamps that mark note sections.
TS_Y4 = re.compile(r"^\s*(\d{2})\.(\d{2})\.(\d{4})\s+(\d{2}):(\d{2})\s*$")
TS_Y2 = re.compile(r"^\s*(\d{2})\.(\d{2})\.(\d{2})\s+(\d{2}):(\d{2})\s*$")
FN_RE = re.compile(r"^Encounter_(?P<eid>.+?)\.txt$", re.IGNORECASE)


def parse_note_timestamp(line):
    """Return a datetime if the line is a note-section timestamp; otherwise None."""
    m = TS_Y4.match(line)
    if m:
        d, mo, y, hh, mm = map(int, m.groups())
        return datetime(y, mo, d, hh, mm)

    m = TS_Y2.match(line)
    if m:
        d, mo, yy, hh, mm = map(int, m.groups())
        y = 2000 + yy if yy <= 69 else 1900 + yy
        return datetime(y, mo, d, hh, mm)

    return None


# 4. Remove note sections documented after the prediction-time cutoff.
def remove_post_cutoff_sections(note_text, cutoff_time):
    """
    Keep note sections whose timestamp is <= cutoff_time.
    Remove sections whose timestamp is > cutoff_time, including all following
    non-timestamped lines until the next timestamped section.
    """
    lines = note_text.splitlines(keepends=True)
    filtered_lines = []
    i = 0

    while i < len(lines):
        ts = parse_note_timestamp(lines[i])

        # Lines without a timestamp are retained unless they belong to
        # a timestamped section that has already been excluded.
        if ts is None:
            filtered_lines.append(lines[i])
            i += 1
            continue

        # Keep timestamped section if it occurred before or at the cutoff.
        if ts <= cutoff_time:
            filtered_lines.append(lines[i])
            i += 1
            while i < len(lines) and parse_note_timestamp(lines[i]) is None:
                filtered_lines.append(lines[i])
                i += 1

        # Otherwise remove this section until the next timestamp.
        else:
            i += 1
            while i < len(lines) and parse_note_timestamp(lines[i]) is None:
                i += 1

    return "".join(filtered_lines)


# 5. Apply filtering to all encounter-level note files.
src_dir = Path("path/to/original_notes")
dst_dir = Path("path/to/timestamp_filtered_notes")
dst_dir.mkdir(parents=True, exist_ok=True)

for path in src_dir.glob("Encounter_*.txt"):
    match = FN_RE.match(path.name)
    if match is None:
        continue

    key = match.group("eid").strip().lower()
    cutoff_time = cutoff_map.get(key)

    if cutoff_time is None:
        continue

    raw_text = path.read_text(encoding="utf-8", errors="ignore")
    filtered_text = remove_post_cutoff_sections(raw_text, cutoff_time)

    # Only filtered text is saved and subsequently provided to the LLM.
    (dst_dir / path.name).write_text(filtered_text, encoding="utf-8")

In [ ]:
# 6. Redundant LLM-based leakage check after timestamp filtering
# Goal: flag adapted clinical notes for manual inspection if the LLM detects
# any remaining documentation timestamp later than the encounter-specific cutoff.
#
# Important: this step does not automatically remove additional text.
# It creates a manual-review list of potentially leaky encounters.

import json
import pandas as pd
from pathlib import Path


def build_llm_leakage_check_prompt(encounter_id, cutoff_time, clinical_note):
    """
    Prompt for the on-premise LLM.
    The LLM is asked only to identify whether any timestamp in the adapted note
    is later than the maximum pre-target recorded_time for that encounter.
    """

    return f"""
You are checking a timestamp-filtered clinical note for possible data leakage.

Encounter ID:
{encounter_id}

Maximum allowed timestamp:
{cutoff_time}

Task:
Review the clinical note below and determine whether it contains any documentation
timestamp that is later than the maximum allowed timestamp.

Only consider explicit documentation timestamps in the note text.
Do not infer timestamps from clinical events unless a timestamp is explicitly written.

Return valid JSON only, with the following fields:
{{
  "flag_for_manual_review": true or false,
  "timestamps_after_cutoff": ["list of timestamps found after cutoff"],
  "largest_timestamp_after_cutoff": "timestamp or null",
  "evidence": "short excerpt showing the timestamp, or null",
  "reason": "brief explanation"
}}

Clinical note:
\"\"\"
{clinical_note}
\"\"\"
"""


def parse_llm_json(response_text):
    """
    Parse JSON returned by the LLM.
    If parsing fails, conservatively flag the note for manual inspection.
    """

    try:
        return json.loads(response_text)
    except Exception:
        return {
            "flag_for_manual_review": True,
            "timestamps_after_cutoff": [],
            "largest_timestamp_after_cutoff": None,
            "evidence": None,
            "reason": "LLM response could not be parsed; flagged conservatively."
        }


def llm_check_for_remaining_post_cutoff_timestamps(
    encounter_id,
    clinical_note,
    cutoff_time,
    llm_client
):
    """
    Run the redundant LLM safety check.

    The object `llm_client` represents the local/on-premise LLM interface.
    It should send the prompt to the local model and return the model response
    as text.
    """

    prompt = build_llm_leakage_check_prompt(
        encounter_id=encounter_id,
        cutoff_time=cutoff_time,
        clinical_note=clinical_note
    )

    response_text = llm_client.generate(prompt)
    result = parse_llm_json(response_text)

    return {
        "encounter_id": encounter_id,
        "recorded_time_max_no_target": cutoff_time,
        "llm_flag_for_manual_review": result.get("flag_for_manual_review", True),
        "llm_timestamps_after_cutoff": result.get("timestamps_after_cutoff", []),
        "llm_largest_timestamp_after_cutoff": result.get(
            "largest_timestamp_after_cutoff", None
        ),
        "llm_evidence": result.get("evidence", None),
        "llm_reason": result.get("reason", None)
    }


# Apply LLM check to the adapted, timestamp-filtered notes.
# `df_notes` contains one row per hospital encounter with:
#   - encounter_id
#   - clinical_notes
#   - recorded_time_max_no_target

llm_review_records = []

for _, row in df_notes.iterrows():
    review_result = llm_check_for_remaining_post_cutoff_timestamps(
        encounter_id=row["encounter_id"],
        clinical_note=row["clinical_notes"],
        cutoff_time=row["recorded_time_max_no_target"],
        llm_client=on_premise_llm
    )

    llm_review_records.append(review_result)


llm_review_df = pd.DataFrame(llm_review_records)

# Encounters requiring manual inspection
manual_review_df = llm_review_df[
    llm_review_df["llm_flag_for_manual_review"] == True
].copy()

manual_review_df.to_csv(
    "manual_review_remaining_post_cutoff_timestamps.csv",
    index=False
)